## Module 6-2 Classification with XGBoost: Predicting the Direction of Earnings Changes

In this session, we aim to replicate the idea of Chen et al. (2022) in a simplified way:

- **Data**: we just use `../data/comp_sample.csv`, instead of the paper's XBRL tags.
- **Model**: `xgboost`, a modern, fast, and widely-used gradient boosting library, instead of the paper's random forest / stochastic gradient boosting implementations.
- **Target**: the same *drift-adjusted* direction of next-year EPS change.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

import xgboost as xgb

### 1. Load and filter the data

In [ ]:
df = pd.read_csv('../data/comp_sample.csv', 
                 dtype={"gvkey": str,})

df = df[
    (df['indfmt'] == 'INDL') &
    (df['curcd'] == 'USD') &
    (df['costat'] == 'A')
].drop_duplicates(subset = ['gvkey', 'fyear']).copy()
df = df.sort_values(['gvkey', 'fyear']).reset_index(drop=True)

print(df.shape)
df.head()

### 2. Build the drift-adjusted earnings-change label

In [ ]:
df['eps'] = (df['ni'] / df['csho']).replace([np.inf, -np.inf], np.nan) # Here we should use IBES Actual EPS. Just a simplified demo for this workshop
df['d_eps'] = df.groupby('gvkey')['eps'].diff()
df['d_eps'] = df['d_eps'].where(df.groupby('gvkey')['fyear'].diff()==1, np.nan)

# Drift: trailing (up to) 4-year average change in EPS, known as of year t
df['drift'] = (
    df.groupby('gvkey')['d_eps']
      .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
)

# Next year's change in EPS, and the drift-adjusted version used for the label
df['d_eps_lead'] = df.groupby('gvkey')['d_eps'].shift(-1).where(df.groupby('gvkey')['fyear'].diff(-1)==-1, np.nan)
df['adj_d_eps_lead'] = df['d_eps_lead'] - df['drift']

df['label'] = np.where(
    df['adj_d_eps_lead'].isna(), np.nan, (df['adj_d_eps_lead'] > 0).astype(np.int8)
)

In [ ]:
# Check how balanced are the two classes
df['label'].value_counts()

### 3. Feature engineering: current value, lagged value, and % change

Dollar-denominated items are scaled by total assets (current value by $Assets_t$, lagged value by $Assets_{t-1}$), matching the paper. Total assets itself, shares outstanding, and share price are per-share/scale items and are kept unscaled, also matching the paper's treatment of "total assets itself and items on a per-share basis."

In [ ]:
ID_COLS = ['gvkey', 'datadate', 'fyear']
LABEL_COLS = ['eps', 'd_eps', 'drift', 'd_eps_lead', 'adj_d_eps_lead', 'label']
NON_FS_COLS = ['au']  # e.g. auditor code: an identifier, not a financial-statement item

# Items that should NOT be scaled by total assets:
NO_SCALE_COLS = ['at', 'csho', 'prcc_f']

exclude_cols = set(ID_COLS + LABEL_COLS + NON_FS_COLS)
numeric_cols = df.select_dtypes(include='number').columns
predictor_base_cols = [c for c in numeric_cols if c not in exclude_cols]

print(f'{len(predictor_base_cols)} raw financial-statement columns auto-detected as predictors:')
print(predictor_base_cols)

In [ ]:
def build_features(data: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Current value, lagged value, and percentage change for each base column."""
    at_cur = data['at']
    at_lag = data.groupby('gvkey')['at'].shift(1).where(data.groupby('gvkey')['fyear'].diff()==1)

    feature_cols = {}
    for col in base_cols:
        cur = data[col]
        lag = data.groupby('gvkey')[col].shift(1).where(data.groupby('gvkey')['fyear'].diff()==1)

        if col in NO_SCALE_COLS:
            cur_scaled, lag_scaled = cur, lag
        else:
            cur_scaled = (cur / at_cur).replace([np.inf, -np.inf], np.nan)
            lag_scaled = (lag / at_lag).replace([np.inf, -np.inf], np.nan)

        pct_change = ((cur - lag) / lag.abs()).replace([np.inf, -np.inf], np.nan)

        feature_cols[f'{col}_cur'] = cur_scaled
        feature_cols[f'{col}_lag'] = lag_scaled
        feature_cols[f'{col}_pctchg'] = pct_change

    return pd.DataFrame(feature_cols, index=data.index)


X_all = build_features(df, predictor_base_cols)
print(X_all.shape)
X_all.head()

In [ ]:
feature_cols = X_all.columns.tolist()

model_df = pd.concat([df[['gvkey', 'fyear', 'label']], X_all], axis=1)
model_df = model_df.dropna(subset=['label']).reset_index(drop=True)

print(model_df.shape)
model_df.groupby('fyear').size()

### 4. Split chronologically, not randomly
- **Train**: earlier fiscal years
- **Validation**: used for XGBoost's early stopping
- **Test**: the most recent fiscal years, held out for performance evaluation

In [ ]:
train = model_df[model_df['fyear'] <= 2019]
val = model_df[(model_df['fyear'] > 2019) & (model_df['fyear'] <= 2021)]
test = model_df[model_df['fyear'] == 2022]

X_train, y_train = train[feature_cols], train['label']
X_val, y_val = val[feature_cols], val['label']
X_test, y_test = test[feature_cols], test['label']

print(f'train: {X_train.shape} ({train.fyear.min()}-{train.fyear.max()})')
print(f'val:   {X_val.shape} ({val.fyear.min()}-{val.fyear.max()})')
print(f'test:  {X_test.shape} ({test.fyear.min()}-{test.fyear.max()})')

### 5. XGBoost

[`xgboost`](https://xgboost.readthedocs.io/) is a fast, regularized implementation of gradient-boosted trees — the same family of model as the stochastic gradient boosting Chen et al. (2022) use, but faster, better-regularized against overfitting, and (critically for us) able to split on missing values natively.

In [ ]:
xgb_tuned = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    importance_type='gain',
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f'Stopped after {xgb_tuned.best_iteration + 1} trees '
      f'(validation AUC = {xgb_tuned.best_score:.3f})')

tuned_test_auc = roc_auc_score(y_test, xgb_tuned.predict_proba(X_test)[:, 1])
print(f'XGBoost (early-stopped) - test AUC: {tuned_test_auc:.3f}')

### 6. Evaluate

We use **ROC-AUC** — the same metric as the paper — because it doesn't depend on picking a probability threshold, and it's directly comparable to the paper's reported numbers (Ou-Penman logit ~ 61.8% vs. their XGBoost-family models ~ 67-69%).

In [ ]:
proba = xgb_tuned.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, proba)
auc = roc_auc_score(y_test, proba)

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random guess (AUC = 0.500)')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Predicting the direction of next-year earnings changes')
plt.legend()
plt.show()

In [ ]:
# A confusion matrix and classification report at the conventional 0.5 cutoff
y_pred = (xgb_tuned.predict_proba(X_test)[:, 1] > 0.5).astype(int)

print('Confusion matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, y_pred))
print("-"*50)
print(classification_report(y_test, y_pred, target_names=['Decrease', 'Increase']))

### 7. Which predictors matter most? (Optional)

XGBoost's built-in `feature_importances_` reports each feature's average **gain**: how much it improved the model's loss function, summed across every split that uses it.

In [ ]:
importances = pd.Series(
    xgb_tuned.feature_importances_, index=feature_cols
).sort_values(ascending=False)

top_n = 20
plt.figure(figsize=(8, 6))
importances.head(top_n).sort_values().plot(kind='barh')
plt.xlabel('XGBoost feature importance (gain)')
plt.title(f'Top {top_n} predictors of earnings-change direction')
plt.tight_layout()
plt.show()

This is useful for a first look, but it can't tell us *how* a feature pushes an individual firm-year's prediction up or down, or which direction the relationship runs. We will talk about this topic in section 6-4o on Explainable AI (`shap`).